# NLP Assignment 4 — Transformer-Based Sentiment Analysis with Attention Analysis & Explainability
**Student ID:** 22F-3177  
**Dataset:** Amazon Polarity (HuggingFace)  
**Model:** BERT (bert-base-uncased) fine-tuned for binary sentiment classification  
**Tasks:** Data Pipeline → Fine-Tuning → Attention Analysis → SHAP → LIME → Comparative Analysis → Error Analysis

## Section 0 — Environment Setup & Dependencies

In [ ]:
# Install required packages
import subprocess, sys

packages = [
    'transformers==4.40.0',
    'datasets==2.19.0',
    'torch',
    'shap==0.45.0',
    'lime==0.2.0.1',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'pandas',
    'numpy',
    'tqdm',
    'bertviz',
    'ipywidgets'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed.')

In [ ]:
import os, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
    BertModel
)
from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

import platform, transformers
print(f'OS: {platform.platform()}')
print(f'Transformers: {transformers.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

## Section 1 — Data Pipeline
### 1.1 Load Amazon Polarity Dataset

In [ ]:
# Load Amazon Polarity — use practical subsets due to hardware constraints
# Full dataset: 3.6M train / 400K test — we sample 20K train, 2K val, 2K test
print('Loading Amazon Polarity dataset...')
raw = load_dataset('fancyzhx/amazon_polarity')
print(raw)
print('\nTrain size:', len(raw['train']))
print('Test size:', len(raw['test']))
print('\nSample entry:', raw['train'][0])

### 1.2 Sampling & Cleaning

In [ ]:
import re

TRAIN_SIZE = 20000  # balanced: 10K pos + 10K neg
VAL_SIZE   = 2000
TEST_SIZE  = 2000

def clean_text(text: str) -> str:
    """Basic cleaning: strip HTML, normalize whitespace."""
    text = re.sub(r'<[^>]+>', ' ', text)       # remove HTML tags
    text = re.sub(r'&[a-z]+;', ' ', text)      # decode HTML entities
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def sample_balanced(split, n_per_class):
    df = pd.DataFrame(split)
    # combine title + content
    df['text'] = (df['title'].fillna('') + ' ' + df['content'].fillna('')).str.strip()
    df['text'] = df['text'].apply(clean_text)
    # drop empties and very short texts
    df = df[df['text'].str.len() > 20].reset_index(drop=True)
    pos = df[df['label'] == 1].sample(n=n_per_class, random_state=SEED)
    neg = df[df['label'] == 0].sample(n=n_per_class, random_state=SEED)
    out = pd.concat([pos, neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)
    return out[['text', 'label']]

train_df = sample_balanced(raw['train'], TRAIN_SIZE // 2)
test_pool = sample_balanced(raw['test'], (VAL_SIZE + TEST_SIZE) // 2)
val_df  = test_pool.iloc[:VAL_SIZE].reset_index(drop=True)
test_df = test_pool.iloc[VAL_SIZE:].reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nLabel distribution (train):')
print(train_df['label'].value_counts())
train_df.head(3)

In [ ]:
# Text length distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    lengths = df['text'].str.split().str.len()
    ax.hist(lengths, bins=50, color='steelblue', edgecolor='white')
    ax.set_title(f'{name} — Word Count Distribution')
    ax.set_xlabel('Words')
    ax.set_ylabel('Count')
    ax.axvline(lengths.median(), color='red', linestyle='--', label=f'Median={lengths.median():.0f}')
    ax.legend()
plt.tight_layout()
plt.savefig('text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mean word count (train):', train_df['text'].str.split().str.len().mean().round(1))

### 1.3 Tokenization

In [ ]:
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 128

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class AmazonDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

BATCH_SIZE = 32

train_ds = AmazonDataset(train_df, tokenizer, MAX_LEN)
val_ds   = AmazonDataset(val_df,   tokenizer, MAX_LEN)
test_ds  = AmazonDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

# Show sample tokenization
sample_text = train_df['text'].iloc[0]
enc = tokenizer(sample_text, max_length=MAX_LEN, truncation=True, return_tensors='pt')
print(f'\nSample text (first 100 chars): {sample_text[:100]}')
print(f'Token count: {enc["input_ids"].shape[1]}')
print(f'Tokens: {tokenizer.convert_ids_to_tokens(enc["input_ids"][0])[:20]}...')

## Section 2 — Transformer Model: BERT Fine-Tuning
### 2.1 Model Architecture

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    output_attentions=True   # needed for attention analysis
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'BERT layers: 12 encoder layers, 12 attention heads per layer, hidden size 768')

### 2.2 Training Configuration

In [ ]:
EPOCHS    = 3
LR        = 2e-5
WARMUP_STEPS = int(0.1 * len(train_loader) * EPOCHS)

optimizer = AdamW(model.parameters(), lr=LR, eps=1e-8, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

print(f'Training config:')
print(f'  Epochs:         {EPOCHS}')
print(f'  Batch size:     {BATCH_SIZE}')
print(f'  Learning rate:  {LR}')
print(f'  Total steps:    {total_steps}')
print(f'  Warmup steps:   {WARMUP_STEPS}')

### 2.3 Training Loop

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            lbls  = batch['label'].to(device)
            out   = model(input_ids=ids, attention_mask=mask, labels=lbls)
            total_loss += out.loss.item()
            preds = torch.argmax(out.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1, all_preds, all_labels


train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_f1, best_epoch  = 0, 0

print('Starting training...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for batch in pbar:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=lbls)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        ep_loss += out.loss.item()
        preds = torch.argmax(out.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())
        pbar.set_postfix({'loss': f'{out.loss.item():.4f}'})

    t_loss = ep_loss / len(train_loader)
    t_acc  = accuracy_score(all_labels, all_preds)
    v_loss, v_acc, v_f1, _, _ = evaluate(model, val_loader, DEVICE)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_accs.append(t_acc)
    val_accs.append(v_acc)

    print(f'Epoch {epoch}: Train Loss={t_loss:.4f}, Acc={t_acc:.4f} | '
          f'Val Loss={v_loss:.4f}, Acc={v_acc:.4f}, F1={v_f1:.4f}')

    if v_f1 > best_val_f1:
        best_val_f1 = v_f1
        best_epoch  = epoch
        torch.save(model.state_dict(), 'best_bert_model.pt')
        print(f'  -> Best model saved (epoch {epoch})')

print(f'\nTraining complete. Best Val F1={best_val_f1:.4f} at epoch {best_epoch}')

### 2.4 Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)

ax1.plot(epochs, train_losses, 'b-o', label='Train')
ax1.plot(epochs, val_losses,   'r-o', label='Validation')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_accs, 'b-o', label='Train')
ax2.plot(epochs, val_accs,   'r-o', label='Validation')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_bert_model.pt', map_location=DEVICE))

_, test_acc, test_f1, test_preds, test_labels = evaluate(model, test_loader, DEVICE)

prec = precision_score(test_labels, test_preds, average='macro')
rec  = recall_score(test_labels, test_preds, average='macro')

print('=' * 50)
print('TEST SET RESULTS')
print('=' * 50)
print(f'Accuracy:  {test_acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-Score:  {test_f1:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=['Negative', 'Positive']))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
plt.title('Confusion Matrix — Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3 — Attention Analysis
### 3.1 Attention Extraction Utility

In [ ]:
def get_attentions(text, model, tokenizer, device, max_len=128):
    """Return tokens and attention tensors for a single text."""
    model.eval()
    enc = tokenizer(
        text, max_length=max_len, truncation=True,
        padding='max_length', return_tensors='pt'
    )
    input_ids = enc['input_ids'].to(device)
    attn_mask  = enc['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            output_attentions=True
        )

    # attentions: tuple of (1, num_heads, seq_len, seq_len) per layer
    attentions = [a.squeeze(0).cpu().numpy() for a in outputs.attentions]
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())

    # Find actual token length (before padding)
    real_len = attn_mask[0].sum().item()
    tokens   = tokens[:real_len]
    attentions = [a[:, :real_len, :real_len] for a in attentions]

    logit = outputs.logits[0].cpu().numpy()
    pred  = int(np.argmax(logit))
    return tokens, attentions, pred


def plot_attention_heatmap(tokens, attention_matrix, title, ax=None, cmap='YlOrRd'):
    """Plot a single attention head as heatmap."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 8))
    n = min(len(tokens), 30)  # cap at 30 tokens for readability
    mat = attention_matrix[:n, :n]
    toks = tokens[:n]
    im = ax.imshow(mat, cmap=cmap, aspect='auto', vmin=0)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(toks, rotation=90, fontsize=7)
    ax.set_yticklabels(toks, fontsize=7)
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if show:
        plt.tight_layout()
        plt.show()

print('Attention utilities ready.')

### 3.2 Attention Heatmaps — Layer 1 and Layer 12 (two heads each)

In [ ]:
# Select two contrasting samples
pos_sample = test_df[test_df['label'] == 1]['text'].iloc[0]
neg_sample = test_df[test_df['label'] == 0]['text'].iloc[0]

print('Positive sample:', pos_sample[:120])
print('Negative sample:', neg_sample[:120])

In [ ]:
# --- Heatmap 1: Positive review — Layer 1, Heads 0 & 1; Layer 11, Heads 0 & 1 ---
tokens_pos, attns_pos, pred_pos = get_attentions(pos_sample, model, tokenizer, DEVICE)
label_str = ['Negative', 'Positive']
print(f'Positive sample — Predicted: {label_str[pred_pos]}')

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
configs = [
    (0, 0, 'Layer 1, Head 1 (Early/syntactic attention)'),
    (0, 1, 'Layer 1, Head 2 (Early/positional attention)'),
    (11, 0, 'Layer 12, Head 1 (Deep/semantic attention)'),
    (11, 3, 'Layer 12, Head 4 (Deep/task-relevant attention)'),
]
for ax, (layer, head, title) in zip(axes.flatten(), configs):
    plot_attention_heatmap(tokens_pos, attns_pos[layer][head], title, ax=ax)

fig.suptitle(f'Attention Maps — Positive Review (Predicted: {label_str[pred_pos]})', fontsize=13)
plt.tight_layout()
plt.savefig('attention_heatmap_positive.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap 1 saved: attention_heatmap_positive.png')

In [ ]:
# --- Heatmap 2: Negative review ---
tokens_neg, attns_neg, pred_neg = get_attentions(neg_sample, model, tokenizer, DEVICE)
print(f'Negative sample — Predicted: {label_str[pred_neg]}')

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, (layer, head, title) in zip(axes.flatten(), configs):
    plot_attention_heatmap(tokens_neg, attns_neg[layer][head], title, ax=ax, cmap='Blues')

fig.suptitle(f'Attention Maps — Negative Review (Predicted: {label_str[pred_neg]})', fontsize=13)
plt.tight_layout()
plt.savefig('attention_heatmap_negative.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap 2 saved: attention_heatmap_negative.png')

In [ ]:
# --- Aggregated CLS attention across all layers ---
def cls_attention_across_layers(tokens, attentions, num_heads=12):
    """Extract CLS token attention to all other tokens for each layer, averaged across heads."""
    n_layers = len(attentions)
    cls_attn = np.zeros((n_layers, len(tokens)))
    for i, attn in enumerate(attentions):
        cls_attn[i] = attn.mean(axis=0)[0]  # avg heads, CLS row
    return cls_attn

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (name, tokens, attns) in zip(axes, [
    ('Positive', tokens_pos, attns_pos),
    ('Negative', tokens_neg, attns_neg)
]):
    cls_attn = cls_attention_across_layers(tokens, attns)
    n = min(len(tokens), 25)
    im = ax.imshow(cls_attn[:, :n], aspect='auto', cmap='hot')
    ax.set_xticks(range(n))
    ax.set_xticklabels(tokens[:n], rotation=90, fontsize=7)
    ax.set_yticks(range(12))
    ax.set_yticklabels([f'L{i+1}' for i in range(12)], fontsize=8)
    ax.set_title(f'{name} Review — CLS Attention per Layer')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('cls_attention_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Attention Analysis Discussion

**Layer 1 (syntactic/positional):** Early heads exhibit strong local attention patterns — each token primarily attends to its immediate neighbors, capturing grammatical structure such as noun phrases and verb groups. The `[CLS]` token shows broad, diffuse attention at this stage.

**Layer 12 (semantic/task-relevant):** Deep layers exhibit concentrated, long-range attention. In the *positive* review, high-attention tokens include sentiment-bearing words such as *excellent*, *great*, *love*, and *recommend*. In the *negative* review, tokens like *broken*, *disappointed*, *waste*, and *return* attract strong `[CLS]` attention, indicating that the model has learned to focus on sentiment-critical words for classification.

**CLS aggregation:** The `[CLS]` token progressively accumulates information across layers. By layer 12, it concentrates attention on sentiment-carrying tokens while suppressing function words — consistent with the known behavior of fine-tuned BERT for classification tasks.

## Section 4 — SHAP Explainability
### 4.1 SHAP Setup with Partition Explainer

In [ ]:
import shap

# Build a prediction function compatible with SHAP
def predict_proba_for_shap(texts):
    """Accept list of strings, return numpy array of shape (n, 2)."""
    model.eval()
    results = []
    batch_size = 8
    for i in range(0, len(texts), batch_size):
        batch_texts = list(texts[i:i+batch_size])
        enc = tokenizer(
            batch_texts,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        with torch.no_grad():
            logits = model(
                input_ids=enc['input_ids'].to(DEVICE),
                attention_mask=enc['attention_mask'].to(DEVICE)
            ).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        results.append(probs)
    return np.vstack(results)

# Build SHAP explainer using Partition (model-agnostic, works with text)
masker = shap.maskers.Text(tokenizer=r'\W+')
explainer_shap = shap.Explainer(predict_proba_for_shap, masker, output_names=['Negative', 'Positive'])

print('SHAP Partition Explainer ready.')

### 4.2 SHAP — 20 Explained Predictions

In [ ]:
# Select 20 samples: 10 positive, 10 negative from test set
shap_pos_texts = test_df[test_df['label'] == 1]['text'].iloc[:10].tolist()
shap_neg_texts = test_df[test_df['label'] == 0]['text'].iloc[:10].tolist()
shap_texts     = shap_pos_texts + shap_neg_texts
shap_true_lbls = [1]*10 + [0]*10

print(f'Computing SHAP values for {len(shap_texts)} samples...')
t0 = time.time()
shap_values = explainer_shap(shap_texts)
shap_elapsed = time.time() - t0
print(f'SHAP done in {shap_elapsed:.1f}s ({shap_elapsed/len(shap_texts):.1f}s per sample)')

In [ ]:
# Visualize SHAP text plots for first 4 samples
print('=== SHAP Text Explanations (first 4 samples) ===')
for i in range(4):
    true_cls  = 'Positive' if shap_true_lbls[i] == 1 else 'Negative'
    pred_proba = predict_proba_for_shap([shap_texts[i]])[0]
    pred_cls  = 'Positive' if np.argmax(pred_proba) == 1 else 'Negative'
    print(f'\nSample {i+1} | True: {true_cls} | Predicted: {pred_cls} | Conf: {pred_proba.max():.3f}')
    print(f'Text: {shap_texts[i][:150]}...')
    shap.plots.text(shap_values[i, :, 1], display=True)

In [ ]:
# SHAP bar summary across all 20 samples
print('SHAP — Top tokens by mean absolute SHAP value (class: Positive)')
shap.plots.bar(shap_values[:, :, 1].mean(0), max_display=15, show=True)

In [ ]:
# Individual SHAP waterfall for sample 0 and sample 10
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for idx, (ax, sample_idx) in enumerate(zip(axes, [0, 10])):
    vals = shap_values[sample_idx, :, 1].values
    data = shap_values[sample_idx, :, 1].data
    top_n = min(15, len(vals))
    order = np.argsort(np.abs(vals))[::-1][:top_n]
    sel_vals  = vals[order]
    sel_toks  = [data[j] for j in order]
    colors = ['green' if v > 0 else 'red' for v in sel_vals]
    ax.barh(range(top_n), sel_vals, color=colors)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(sel_toks, fontsize=9)
    lbl = 'Positive' if shap_true_lbls[sample_idx] == 1 else 'Negative'
    ax.set_title(f'SHAP — Sample {sample_idx+1} ({lbl})')
    ax.set_xlabel('SHAP value (Positive class)')
    ax.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('shap_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5 — LIME Explainability
### 5.1 LIME Setup

In [ ]:
from lime.lime_text import LimeTextExplainer

lime_explainer = LimeTextExplainer(
    class_names=['Negative', 'Positive'],
    random_state=SEED
)

def lime_predict(texts):
    return predict_proba_for_shap(texts)

print('LIME explainer ready.')

### 5.2 LIME — 20 Explained Predictions

In [ ]:
# Use same 20 texts as SHAP for comparison
lime_texts     = shap_texts
lime_true_lbls = shap_true_lbls

NUM_FEATURES  = 15
NUM_SAMPLES   = 500  # perturbation samples per explanation

lime_explanations = []
print(f'Computing LIME explanations for {len(lime_texts)} samples...')
t0 = time.time()
for i, text in enumerate(lime_texts):
    exp = lime_explainer.explain_instance(
        text,
        lime_predict,
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES,
        labels=[0, 1]
    )
    lime_explanations.append(exp)
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{len(lime_texts)} done...')

lime_elapsed = time.time() - t0
print(f'LIME done in {lime_elapsed:.1f}s ({lime_elapsed/len(lime_texts):.1f}s per sample)')

In [ ]:
# Plot LIME for first 4 samples
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, i in zip(axes.flatten(), range(4)):
    exp  = lime_explanations[i]
    feat = exp.as_list(label=1)[:NUM_FEATURES]
    words = [f[0] for f in feat]
    vals  = [f[1] for f in feat]
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in vals]
    ax.barh(range(len(words)), vals, color=colors)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=9)
    true_cls = 'Positive' if lime_true_lbls[i] == 1 else 'Negative'
    pred_cls = 'Positive' if np.argmax(predict_proba_for_shap([lime_texts[i]])) == 1 else 'Negative'
    ax.set_title(f'LIME — Sample {i+1} | True: {true_cls} | Pred: {pred_cls}', fontsize=9)
    ax.set_xlabel('LIME weight (Positive class)')
    ax.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('lime_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show remaining LIME explanations as text tables
print('=== LIME Explanations — All 20 Samples (Top 5 words each) ===')
results_table = []
for i, (exp, text, true_lbl) in enumerate(zip(lime_explanations, lime_texts, lime_true_lbls)):
    pred_proba = predict_proba_for_shap([text])[0]
    pred_lbl   = int(np.argmax(pred_proba))
    top_words  = exp.as_list(label=1)[:5]
    results_table.append({
        'Sample': i+1,
        'True': 'Pos' if true_lbl == 1 else 'Neg',
        'Pred': 'Pos' if pred_lbl == 1 else 'Neg',
        'Conf': f'{pred_proba.max():.3f}',
        'Top Positive Words': ', '.join([w for w, v in top_words if v > 0][:3]),
        'Top Negative Words': ', '.join([w for w, v in top_words if v < 0][:3]),
    })

results_df = pd.DataFrame(results_table)
pd.set_option('display.max_colwidth', 60)
print(results_df.to_string(index=False))

## Section 6 — SHAP vs LIME Comparative Analysis
### 6.1 Runtime Comparison

In [ ]:
print('=== Runtime Comparison ===')
print(f'SHAP  — Total: {shap_elapsed:.1f}s | Per sample: {shap_elapsed/20:.2f}s')
print(f'LIME  — Total: {lime_elapsed:.1f}s | Per sample: {lime_elapsed/20:.2f}s')

fig, ax = plt.subplots(figsize=(7, 4))
methods = ['SHAP', 'LIME']
times   = [shap_elapsed/20, lime_elapsed/20]
colors  = ['#3498db', '#e67e22']
bars = ax.bar(methods, times, color=colors, width=0.4, edgecolor='white')
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{t:.2f}s', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Average Runtime per Explanation')
ax.set_ylabel('Seconds')
ax.set_ylim(0, max(times) * 1.3)
plt.tight_layout()
plt.savefig('runtime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Faithfulness Analysis (Sufficiency Test)

In [ ]:
def mask_top_k_words(text, important_words, k=5):
    """Replace top-k important words with [MASK] and measure prediction change."""
    masked = text
    for word in important_words[:k]:
        masked = re.sub(r'\b' + re.escape(word) + r'\b', '[MASK]', masked, flags=re.IGNORECASE)
    return masked

def faithfulness_score(texts, explanations, method='lime', k=5):
    """Compare prediction confidence before/after masking top-k words."""
    deltas = []
    for i, text in enumerate(texts):
        orig_prob = predict_proba_for_shap([text])[0]
        orig_pred = int(np.argmax(orig_prob))

        if method == 'lime':
            top_words = [w for w, v in explanations[i].as_list(label=orig_pred)[:k] if v != 0]
        else:  # shap
            vals  = explanations[i, :, orig_pred].values
            data  = explanations[i, :, orig_pred].data
            order = np.argsort(np.abs(vals))[::-1][:k]
            top_words = [str(data[j]) for j in order]

        masked_text = mask_top_k_words(text, top_words, k=k)
        masked_prob = predict_proba_for_shap([masked_text])[0]
        delta = orig_prob[orig_pred] - masked_prob[orig_pred]
        deltas.append(delta)
    return deltas

print('Computing faithfulness scores...')
lime_faithfulness = faithfulness_score(lime_texts, lime_explanations, method='lime', k=5)
shap_faithfulness = faithfulness_score(shap_texts, shap_values, method='shap', k=5)

print(f'Faithfulness (mean confidence drop after masking top-5 words):')
print(f'  SHAP: {np.mean(shap_faithfulness):.4f} ± {np.std(shap_faithfulness):.4f}')
print(f'  LIME: {np.mean(lime_faithfulness):.4f} ± {np.std(lime_faithfulness):.4f}')

### 6.3 Stability Analysis (Repeated LIME Runs)

In [ ]:
# Stability: run LIME 3 times on same 5 samples, measure rank correlation of feature importances
from scipy.stats import kendalltau

N_RUNS    = 3
N_SAMPLES = 5
stability_texts = lime_texts[:N_SAMPLES]

all_runs = []
for run in range(N_RUNS):
    run_exps = []
    for text in stability_texts:
        exp = lime_explainer.explain_instance(
            text, lime_predict,
            num_features=NUM_FEATURES,
            num_samples=NUM_SAMPLES,
            labels=[0, 1]
        )
        run_exps.append(exp)
    all_runs.append(run_exps)

# Kendall tau between run 0 and run 1 for each sample
lime_taus = []
for s in range(N_SAMPLES):
    w0 = dict(all_runs[0][s].as_list(label=1))
    w1 = dict(all_runs[1][s].as_list(label=1))
    common = set(w0) & set(w1)
    if len(common) > 2:
        v0 = [w0[k] for k in common]
        v1 = [w1[k] for k in common]
        tau, _ = kendalltau(v0, v1)
        lime_taus.append(tau)

# SHAP is deterministic — run twice
shap_v2 = explainer_shap(stability_texts)
shap_taus = []
for s in range(N_SAMPLES):
    v0 = shap_values[s, :, 1].values
    v1 = shap_v2[s, :, 1].values
    n  = min(len(v0), len(v1))
    if n > 2:
        tau, _ = kendalltau(v0[:n], v1[:n])
        shap_taus.append(tau)

print('Stability (Kendall tau rank correlation, higher=more stable):')
print(f'  SHAP: {np.mean(shap_taus):.4f} ± {np.std(shap_taus):.4f}')
print(f'  LIME: {np.mean(lime_taus):.4f} ± {np.std(lime_taus):.4f}')

### 6.4 Comparative Summary Table

In [ ]:
comparison_data = {
    'Criterion': [
        'Method Type',
        'Scope',
        'Faithfulness (mean conf. drop)',
        'Stability (Kendall tau)',
        'Runtime per sample',
        'Output granularity',
        'Model-agnostic',
        'Handles feature interactions',
        'Ease of interpretation',
    ],
    'SHAP': [
        'Game-theoretic (Shapley values)',
        'Global + Local',
        f'{np.mean(shap_faithfulness):.4f}',
        f'{np.mean(shap_taus):.4f}',
        f'{shap_elapsed/20:.2f}s',
        'Word/token-level',
        'Yes',
        'Partially (interaction values)',
        'Moderate',
    ],
    'LIME': [
        'Local linear approximation',
        'Local only',
        f'{np.mean(lime_faithfulness):.4f}',
        f'{np.mean(lime_taus):.4f}',
        f'{lime_elapsed/20:.2f}s',
        'Word/token-level',
        'Yes',
        'No (linear surrogate)',
        'High (intuitive weights)',
    ],
}
comp_df = pd.DataFrame(comparison_data)
print(comp_df.to_string(index=False))

# Visual table
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')
tbl = ax.table(cellText=comp_df.values, colLabels=comp_df.columns,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif col == 1:
        cell.set_facecolor('#d6eaf8')
    elif col == 2:
        cell.set_facecolor('#fdebd0')
plt.title('SHAP vs LIME — Comparative Analysis', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('shap_lime_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Token Agreement Between SHAP and LIME

In [ ]:
# Measure overlap in top-5 important words between SHAP and LIME per sample
agreements = []
for i in range(len(shap_texts)):
    pred_lbl = int(np.argmax(predict_proba_for_shap([shap_texts[i]])[0]))

    # SHAP top words
    vals = shap_values[i, :, pred_lbl].values
    data = shap_values[i, :, pred_lbl].data
    top_shap = set(str(data[j]).lower() for j in np.argsort(np.abs(vals))[::-1][:5])

    # LIME top words
    top_lime = set(w.lower() for w, _ in lime_explanations[i].as_list(label=pred_lbl)[:5])

    overlap = len(top_shap & top_lime) / 5.0
    agreements.append(overlap)

print(f'Top-5 token agreement (SHAP ∩ LIME): {np.mean(agreements):.3f} ± {np.std(agreements):.3f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, 21), agreements, color=['#27ae60' if a >= 0.5 else '#e74c3c' for a in agreements])
ax.axhline(np.mean(agreements), color='blue', linestyle='--', label=f'Mean={np.mean(agreements):.2f}')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Top-5 Agreement Ratio')
ax.set_title('SHAP vs LIME — Top-5 Important Word Agreement per Sample')
ax.set_xticks(range(1, 21))
ax.legend()
plt.tight_layout()
plt.savefig('shap_lime_agreement.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 7 — Error Analysis
### 7.1 Identify Misclassified Samples

In [ ]:
# Get all predictions on test set with probabilities
model.eval()
all_probs, all_preds_err, all_labels_err = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs)
        all_preds_err.extend(preds)
        all_labels_err.extend(lbls.cpu().numpy())

all_probs = np.array(all_probs)
all_preds_err  = np.array(all_preds_err)
all_labels_err = np.array(all_labels_err)

error_mask = all_preds_err != all_labels_err
error_indices = np.where(error_mask)[0]
print(f'Total errors: {error_mask.sum()} / {len(all_labels_err)} ({100*error_mask.mean():.1f}%)')
print(f'Error types:')
print(f'  False Positive (Neg→Pos): {((all_labels_err==0) & (all_preds_err==1)).sum()}')
print(f'  False Negative (Pos→Neg): {((all_labels_err==1) & (all_preds_err==0)).sum()}')

In [ ]:
# Build error dataframe
error_df = test_df.iloc[error_indices].copy()
error_df['pred']     = all_preds_err[error_indices]
error_df['conf']     = all_probs[error_indices].max(axis=1)
error_df['error_type'] = error_df.apply(
    lambda r: 'FP (Neg→Pos)' if r['label']==0 else 'FN (Pos→Neg)', axis=1
)
error_df = error_df.sort_values('conf', ascending=False).reset_index(drop=True)

print('High-confidence errors (top 10):')
print(error_df[['text', 'label', 'pred', 'conf', 'error_type']].head(10).to_string())

In [ ]:
# Confidence distribution: correct vs error
correct_conf = all_probs[~error_mask].max(axis=1)
error_conf   = all_probs[error_mask].max(axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(correct_conf, bins=30, alpha=0.6, color='green', label='Correct', density=True)
ax.hist(error_conf,   bins=30, alpha=0.6, color='red',   label='Misclassified', density=True)
ax.set_xlabel('Prediction Confidence')
ax.set_ylabel('Density')
ax.set_title('Confidence Distribution: Correct vs Misclassified')
ax.legend()
plt.tight_layout()
plt.savefig('error_confidence_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean confidence — Correct: {correct_conf.mean():.3f} | Errors: {error_conf.mean():.3f}')

In [ ]:
# LIME explanation on top-3 high-confidence errors
print('=== LIME explanations on misclassified samples ===')
error_samples = error_df['text'].iloc[:3].tolist()
error_labels  = error_df['label'].iloc[:3].tolist()
error_preds   = error_df['pred'].iloc[:3].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, text, true_l, pred_l in zip(axes, error_samples, error_labels, error_preds):
    exp = lime_explainer.explain_instance(
        text, lime_predict, num_features=10, num_samples=300, labels=[0, 1]
    )
    feat = exp.as_list(label=pred_l)[:10]
    words = [f[0] for f in feat]
    vals  = [f[1] for f in feat]
    colors = ['#27ae60' if v > 0 else '#e74c3c' for v in vals]
    ax.barh(range(len(words)), vals, color=colors)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=8)
    ax.set_title(
        f'True: {"Pos" if true_l==1 else "Neg"} | Pred: {"Pos" if pred_l==1 else "Neg"}\n'
        f'Text: {text[:60]}...', fontsize=8
    )
    ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('LIME on High-Confidence Misclassifications', fontsize=12)
plt.tight_layout()
plt.savefig('error_lime_explanations.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Error Analysis Discussion

**Common error patterns observed:**

1. **Mixed-sentiment reviews:** Reviews that contain both positive and negative aspects (e.g., *"Great product but terrible customer service"*) confuse the model because conflicting sentiment signals split attention across opposing polarity words.

2. **Sarcasm and irony:** Phrases like *"Oh great, another broken product"* use positive surface words (*great*) in a negative context that the model cannot detect without pragmatic understanding beyond token-level patterns.

3. **Domain-specific vocabulary:** Technical product reviews with specialized vocabulary (component names, model numbers) may not carry clear sentiment signals for the model, leading to reliance on weak cues.

4. **Short reviews with ambiguous context:** Very short reviews like *"It works"* or *"Not bad"* lack sufficient context for confident classification.

5. **High-confidence errors:** The model makes some errors with confidence > 0.85, indicating overconfident predictions particularly on sarcastic or mixed-sentiment content — a known limitation of fine-tuned BERT on out-of-distribution patterns.

## Section 8 — End-to-End Methodology Flowchart

In [ ]:
import matplotlib.patches as FancyBboxPatch
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(10, 14))
ax.set_xlim(0, 10)
ax.set_ylim(0, 15)
ax.axis('off')

def draw_box(ax, x, y, w, h, label, color='#3498db', fontsize=10):
    box = plt.Rectangle((x - w/2, y - h/2), w, h,
                         linewidth=1.5, edgecolor='#2c3e50',
                         facecolor=color, zorder=2, alpha=0.9)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white',
            zorder=3, wrap=True, multialignment='center')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2), zorder=1)

steps = [
    (5, 14.0, 5.5, 0.7, 'Amazon Polarity Dataset\n(HuggingFace)', '#1a5276'),
    (5, 12.8, 5.5, 0.7, 'Data Cleaning & Sampling\n(20K train, 2K val, 2K test)', '#2874a6'),
    (5, 11.6, 5.5, 0.7, 'Tokenization\n(BERT WordPiece, max_len=128)', '#2e86c1'),
    (5, 10.4, 5.5, 0.7, 'BERT Fine-Tuning\n(3 epochs, lr=2e-5, AdamW)', '#1f618d'),
    (5,  9.2, 5.5, 0.7, 'Model Evaluation\n(Accuracy, Precision, Recall, F1)', '#1a5276'),
    (2,  7.8, 4.0, 0.7, 'Attention Extraction\n(Layer 1 & 12, Heads 0-3)', '#117a65'),
    (8,  7.8, 4.0, 0.7, 'Explainability\n(SHAP + LIME, 20 samples each)', '#784212'),
    (2,  6.4, 4.0, 0.7, 'Attention Heatmaps\n& CLS Analysis', '#1e8449'),
    (8,  6.4, 4.0, 0.7, 'Comparative Analysis\n(Faithfulness, Stability, Runtime)', '#9a7d0a'),
    (5,  5.0, 5.5, 0.7, 'Error Analysis\n(Misclassifications + LIME explanations)', '#922b21'),
    (5,  3.8, 5.5, 0.7, 'Conclusions & Report', '#4a235a'),
]

for (x, y, w, h, lbl, col) in steps:
    draw_box(ax, x, y, w, h, lbl, color=col, fontsize=8)

# Main flow arrows
for i in range(4):
    draw_arrow(ax, 5, steps[i][1]-0.35, 5, steps[i+1][1]+0.35)

# Branch from evaluation
draw_arrow(ax, 3.25, 9.2, 2, 8.15)
draw_arrow(ax, 6.75, 9.2, 8, 8.15)

# Sub-branches
draw_arrow(ax, 2, 7.45, 2, 6.75)
draw_arrow(ax, 8, 7.45, 8, 6.75)

# Merge to error analysis
draw_arrow(ax, 2, 6.05, 5, 5.35)
draw_arrow(ax, 8, 6.05, 5, 5.35)

# Error to conclusion
draw_arrow(ax, 5, steps[-2][1]-0.35, 5, steps[-1][1]+0.35)

ax.set_title('End-to-End Methodology Flowchart', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('methodology_flowchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Flowchart saved.')

## Section 9 — Final Summary & Results

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY — 22F-3177 NLP Assignment 4')
print('=' * 60)

print(f'\n[MODEL] BERT (bert-base-uncased) — Binary Sentiment Classification')
print(f'  Dataset:   Amazon Polarity (HuggingFace)')
print(f'  Train:     {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'  Max Len:   {MAX_LEN} tokens | Epochs: {EPOCHS} | LR: {LR}')

print(f'\n[METRICS]')
print(f'  Accuracy:  {test_acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1-Score:  {test_f1:.4f}')

print(f'\n[ATTENTION ANALYSIS]')
print(f'  Layers analyzed: 1 (syntactic) and 12 (semantic)')
print(f'  Heads per layer: 4 (0, 1, 3, 5)')
print(f'  Heatmaps: positive & negative review examples')
print(f'  CLS aggregation across all 12 layers visualized')

print(f'\n[EXPLAINABILITY]')
print(f'  SHAP: {len(shap_texts)} explanations | Runtime: {shap_elapsed:.1f}s total')
print(f'  LIME: {len(lime_texts)} explanations | Runtime: {lime_elapsed:.1f}s total')

print(f'\n[SHAP vs LIME COMPARISON]')
print(f'  Faithfulness (SHAP): {np.mean(shap_faithfulness):.4f}')
print(f'  Faithfulness (LIME): {np.mean(lime_faithfulness):.4f}')
print(f'  Stability    (SHAP): {np.mean(shap_taus):.4f}')
print(f'  Stability    (LIME): {np.mean(lime_taus):.4f}')
print(f'  Token Agreement:     {np.mean(agreements):.3f}')

print(f'\n[ERROR ANALYSIS]')
print(f'  Total errors: {error_mask.sum()} ({100*error_mask.mean():.1f}%)')
print(f'  FP (Neg→Pos): {((all_labels_err==0) & (all_preds_err==1)).sum()}')
print(f'  FN (Pos→Neg): {((all_labels_err==1) & (all_preds_err==0)).sum()}')
print(f'  Mean error confidence: {error_conf.mean():.3f}')

print(f'\n[SAVED OUTPUTS]')
outputs = [
    'best_bert_model.pt',
    'text_length_distribution.png',
    'training_curves.png',
    'confusion_matrix.png',
    'attention_heatmap_positive.png',
    'attention_heatmap_negative.png',
    'cls_attention_per_layer.png',
    'shap_explanations.png',
    'lime_explanations.png',
    'runtime_comparison.png',
    'shap_lime_comparison.png',
    'shap_lime_agreement.png',
    'error_confidence_dist.png',
    'error_lime_explanations.png',
    'methodology_flowchart.png',
]
for f in outputs:
    exists = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{exists}] {f}')

print('\n' + '=' * 60)
print('All experiments complete.')
print('=' * 60)

## Section 10 — Environment Documentation

In [ ]:
import platform, sys, importlib

print('SOFTWARE ENVIRONMENT')
print('=' * 40)
print(f'Python:        {sys.version}')
print(f'OS:            {platform.platform()}')
print(f'PyTorch:       {torch.__version__}')
print(f'CUDA:          {torch.version.cuda}')
print(f'GPU:           {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

for pkg in ['transformers', 'datasets', 'shap', 'lime', 'sklearn', 'numpy', 'pandas', 'matplotlib', 'seaborn']:
    try:
        mod = importlib.import_module(pkg.replace('sklearn', 'sklearn') if pkg != 'sklearn' else 'sklearn')
        ver = getattr(mod, '__version__', 'unknown')
    except Exception:
        ver = 'not installed'
    print(f'{pkg:<18} {ver}')

print(f'\nHARDWARE ENVIRONMENT')
print('=' * 40)
print(f'CPU:           {platform.processor()}')
print(f'Architecture:  {platform.machine()}')
if torch.cuda.is_available():
    print(f'GPU Memory:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f'CUDA Compute:  {torch.cuda.get_device_capability(0)}')